# 02 Exploratory Data Analysis

Deep dive into the 3 prepared Parquet files to understand patterns, correlations, and anomalies before modeling.

**Goals:**
- Understand uncertainty parameter space and how it drives production
- Visualize production curve families and identify clusters
- Profile grid property distributions across cases
- Validate train/val/test split consistency
- Identify features and patterns that inform model architecture

In [0]:
%pip install pandas numpy pyarrow matplotlib seaborn scipy -q

In [0]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import ks_2samp
from IPython.display import display

_curr = Path.cwd().resolve()
REPO_ROOT = None
temp = _curr
while temp != temp.parent:
    if (temp / "Anas").exists() and (temp / "data").exists():
        REPO_ROOT = temp
        break
    temp = temp.parent

if REPO_ROOT is not None:
    DATA_DIR = REPO_ROOT / "data" / "shared"
    EDA_DIR = REPO_ROOT / "Anas" / "outputs" / "eda"
else:
    ROOT = Path("/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP")
    DATA_DIR = ROOT / "data"
    EDA_DIR = ROOT / "outputs" / "eda"

FIG_DIR = EDA_DIR / "figures"
TABLE_DIR = EDA_DIR / "tables"

for directory in [FIG_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

UNC_PATH = DATA_DIR / "uncertainty_params.parquet"
PROD_PATH = DATA_DIR / "production_timeseries.parquet"
GRID_PATH = DATA_DIR / "grid_properties.parquet"

for path in [UNC_PATH, PROD_PATH, GRID_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Run notebook 01 first; missing {path}")

PARAMS = [
    "Fault Transmissibility",
    "Porosity Multiplier",
    "Permeability Multiplier",
    "Aquifer Pore Volume",
]
SPLITS = ["train", "validation", "test"]
COLORS = {
    "train": "#2878B5",
    "validation": "#E69F00",
    "test": "#009E73",
    "history": "#202020",
}
METRICS = ["oil_cum", "gas_cum", "water_cum",
           "oil_rate", "gas_rate", "water_rate"]
LABELS = {
    "oil_cum": "Cumulative oil (million STB)",
    "gas_cum": "Cumulative gas (million MSCF)",
    "water_cum": "Cumulative water (million STB)",
    "oil_rate": "Oil rate (STB/day)",
    "gas_rate": "Gas rate (MSCF/day)",
    "water_rate": "Water rate (STB/day)",
}
SEED = 42

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def show_save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

def save_table(frame, name, index=False):
    frame.to_csv(TABLE_DIR / f"{name}.csv", index=index)

def split_for_case(case_num):
    case_num = int(case_num)
    if case_num == 0:
        return "history"
    if 1 <= case_num <= 70:
        return "train"
    if 71 <= case_num <= 85:
        return "validation"
    if 86 <= case_num <= 100:
        return "test"
    return "unknown"

def sample_curve(series, max_points=500):
    if len(series) <= max_points:
        return series
    positions = np.unique(
        np.linspace(0, len(series) - 1, max_points).astype(int)
    )
    return series.iloc[positions]

def interpolate_inside(series, dates):
    dates = pd.DatetimeIndex(dates)
    result = np.full(len(dates), np.nan)
    clean = series.dropna().sort_index()
    if len(clean) < 2:
        return result
    source_x = clean.index.asi8.astype(np.float64)
    target_x = dates.asi8.astype(np.float64)
    inside = (target_x >= source_x[0]) & (target_x <= source_x[-1])
    result[inside] = np.interp(
        target_x[inside], source_x, clean.to_numpy(dtype=float)
    )
    return result

print(f"Input directory: {DATA_DIR}")
print(f"Figures: {FIG_DIR}")
print(f"Tables: {TABLE_DIR}")

In [0]:
unc = pd.read_parquet(UNC_PATH)
required_unc = {"case_num", "case_id", "split", *PARAMS}
missing = required_unc - set(unc.columns)
if missing:
    raise ValueError(f"Missing uncertainty columns: {sorted(missing)}")

unc["case_num"] = pd.to_numeric(unc["case_num"], errors="raise").astype(int)
unc = unc.sort_values("case_num").reset_index(drop=True)

if unc["case_num"].duplicated().any():
    raise ValueError("uncertainty_params contains duplicate case_num values.")

unc["expected_split"] = unc["case_num"].map(split_for_case)
split_errors = unc.loc[unc["split"] != unc["expected_split"]].copy()
save_table(split_errors, "uncertainty_split_errors")
display(unc.groupby("split")[PARAMS].agg(["count", "min", "mean", "max"]))

prod = pd.read_parquet(
    PROD_PATH, columns=["date", "value", "metric", "unit", "case_num", "split"]
)
raw_prod_rows = len(prod)
prod["date"] = pd.to_datetime(prod["date"], errors="coerce")
prod["value"] = pd.to_numeric(prod["value"], errors="coerce")

valid_case = (
    prod["case_num"].notna()
    & pd.to_numeric(prod["case_num"], errors="coerce").between(0, 100)
    & (pd.to_numeric(prod["case_num"], errors="coerce") % 1 == 0)
)
invalid_dates = int(prod["date"].isna().sum())
invalid_values = int((~np.isfinite(prod["value"].to_numpy(dtype=float))).sum())
invalid_cases = int((~valid_case).sum())

prod = prod.loc[valid_case & prod["date"].notna()].copy()
prod["case_num"] = prod["case_num"].astype(int)
split_mismatch_rows = int(
    (prod["split"] != prod["case_num"].map(split_for_case)).sum()
)
unit_audit = prod.groupby("metric")["unit"].unique().apply(list)
display(unit_audit.to_frame("units"))

prod["metric"] = prod["metric"].astype("category")

curves = {}
audit_rows = []
conflict_examples = []

for (case_num, metric), group in prod.groupby(
    ["case_num", "metric"], observed=True, sort=True
):
    group = group.sort_values("date")
    finite = np.isfinite(group["value"].to_numpy(dtype=float))
    good = group.loc[finite, ["date", "value"]]

    grouped = good.groupby("date", sort=True)["value"]
    stats = grouped.agg(["first", "min", "max", "size"])
    conflict = stats["min"] != stats["max"]

    if conflict.any():
        examples = stats.loc[conflict].head(10).reset_index()
        examples["case_num"] = int(case_num)
        examples["metric"] = str(metric)
        conflict_examples.append(examples)

    series = stats.loc[~conflict, "first"].astype(float)
    series.name = str(metric)
    curves[(int(case_num), str(metric))] = series

    unique_dates = pd.DatetimeIndex(group["date"].unique()).sort_values()
    gaps_days = np.diff(unique_dates.asi8) / (86400 * 1e9)
    clean_diff = series.diff().dropna()

    audit_rows.append({
        "case_num": int(case_num),
        "metric": str(metric),
        "split": split_for_case(case_num),
        "raw_rows": len(group),
        "invalid_value_rows": int((~finite).sum()),
        "unique_timestamps": len(unique_dates),
        "duplicate_extra_rows": len(group) - len(unique_dates),
        "conflicting_timestamps": int(conflict.sum()),
        "usable_timestamps": len(series),
        "start": unique_dates.min() if len(unique_dates) else pd.NaT,
        "end": unique_dates.max() if len(unique_dates) else pd.NaT,
        "median_step_days": float(np.median(gaps_days)) if len(gaps_days) else np.nan,
        "min_step_days": float(np.min(gaps_days)) if len(gaps_days) else np.nan,
        "max_step_days": float(np.max(gaps_days)) if len(gaps_days) else np.nan,
        "negative_values": int((series < -1e-8).sum()),
        "cumulative_decreases": (
            int((clean_diff < -1e-6).sum())
            if str(metric).endswith("_cum") else 0
        ),
    })

production_audit = pd.DataFrame(audit_rows)
save_table(production_audit, "production_quality_by_case_metric")

conflicts = (
    pd.concat(conflict_examples, ignore_index=True)
    if conflict_examples else
    pd.DataFrame(columns=["date", "first", "min", "max", "size", "case_num", "metric"])
)
save_table(conflicts, "conflicting_timestamp_examples")

file_inventory = pd.DataFrame([
    {
        "file": path.name,
        "rows": pq.ParquetFile(path).metadata.num_rows,
        "columns": len(pq.read_schema(path).names),
        "size_MB": path.stat().st_size / 1e6,
    }
    for path in [UNC_PATH, PROD_PATH, GRID_PATH]
])
display(file_inventory)
save_table(file_inventory, "file_inventory")

print({
    "raw_production_rows": raw_prod_rows,
    "invalid_dates": invalid_dates,
    "invalid_numeric_values": invalid_values,
    "invalid_case_rows": invalid_cases,
    "production_split_mismatch_rows": split_mismatch_rows,
})
display(production_audit.groupby("metric").agg(
    cases=("case_num", "nunique"),
    first_timestamp=("start", "min"),
    last_timestamp=("end", "max"),
    median_sampling_days=("median_step_days", "median"),
    duplicate_extra_rows=("duplicate_extra_rows", "sum"),
    conflicting_timestamps=("conflicting_timestamps", "sum"),
    cumulative_decreases=("cumulative_decreases", "sum"),
))

if len(conflicts):
    warnings.warn(
        "Conflicting timestamps exist. Resolve their source before modeling; "
        "EDA below excludes those timestamps."
    )

del prod

In [0]:
display(unc[PARAMS].describe().T)
save_table(unc[PARAMS].describe().T, "uncertainty_descriptive_statistics", index=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, param in zip(axes.flat, PARAMS):
    sns.histplot(
        data=unc, x=param, hue="split",
        hue_order=SPLITS, palette=COLORS,
        bins=12, element="step", stat="density",
        common_norm=False, fill=False, ax=ax,
    )
    ax.set_title(param)
    ax.set_xlabel("Parameter value")
    ax.set_ylabel("Within-split density")
fig.suptitle("Uncertainty parameter coverage", fontsize=17)
fig.tight_layout()
show_save(fig, "01_parameter_distributions")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
        unc[PARAMS].corr(method=method),
        vmin=-1, vmax=1, cmap="vlag", annot=True, fmt=".2f",
        square=True, ax=ax, cbar_kws={"label": "Correlation"},
    )
    ax.set_title(f"{method.title()} correlation")
fig.tight_layout()
show_save(fig, "02_parameter_correlations")

pair = sns.pairplot(
    unc, vars=PARAMS, hue="split",
    hue_order=SPLITS, palette=COLORS,
    diag_kind="hist",
    plot_kws={"s": 35, "alpha": 0.8},
    diag_kws={"bins": 10, "alpha": 0.35},
    height=2.3,
)
pair.fig.suptitle("Parameter-space design and split overlap", y=1.02)
show_save(pair.fig, "03_parameter_space")

In [0]:
from scipy.spatial.distance import cdist

train_unc = unc.loc[unc["split"] == "train"].copy()

shift_rows = []
for param in PARAMS:
    train_values = train_unc[param].dropna()
    train_min, train_max = train_values.min(), train_values.max()
    train_std = train_values.std(ddof=1)

    for split in ["validation", "test"]:
        held = unc.loc[unc["split"] == split, param].dropna()
        if len(train_values) and len(held):
            test_result = ks_2samp(train_values, held)
            shift_rows.append({
                "parameter": param,
                "split": split,
                "train_min": train_min,
                "train_max": train_max,
                "heldout_min": held.min(),
                "heldout_max": held.max(),
                "outside_train_range": int(
                    ((held < train_min) | (held > train_max)).sum()
                ),
                "mean_shift_in_train_SD": (
                    (held.mean() - train_values.mean()) / train_std
                    if train_std > 0 else np.nan
                ),
                "KS_statistic": test_result.statistic,
                "KS_pvalue_unadjusted": test_result.pvalue,
            })

shift_report = pd.DataFrame(shift_rows)
display(shift_report)
save_table(shift_report, "parameter_split_shift")

center = train_unc[PARAMS].mean()
scale = train_unc[PARAMS].std(ddof=1).replace(0, 1)
train_z = ((train_unc[PARAMS] - center) / scale).to_numpy()
all_z = ((unc[PARAMS] - center) / scale).to_numpy()
distances = cdist(all_z, train_z)

train_ids = train_unc["case_num"].to_numpy()
all_ids = unc["case_num"].to_numpy()
distances[all_ids[:, None] == train_ids[None, :]] = np.inf

nearest = np.argmin(distances, axis=1)
coverage = unc[["case_num", "split"]].copy()
coverage["nearest_train_case"] = train_ids[nearest]
coverage["nearest_train_distance"] = distances[
    np.arange(len(unc)), nearest
]
display(coverage.sort_values("nearest_train_distance", ascending=False).head(15))
save_table(coverage, "parameter_space_nearest_training_case")

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=coverage, x="split", y="nearest_train_distance",
    order=SPLITS, color="#E5E5E5", ax=ax,
)
sns.stripplot(
    data=coverage, x="split", y="nearest_train_distance",
    order=SPLITS, hue="split", palette=COLORS,
    jitter=0.15, size=5, ax=ax,
)
if ax.get_legend() is not None:
    ax.get_legend().remove()
ax.set(
    title="Distance to the nearest training parameter combination",
    xlabel="",
    ylabel="Euclidean distance in training-standardized inputs",
)
fig.tight_layout()
show_save(fig, "04_split_parameter_coverage")

In [0]:
legend_handles = [
    Line2D([0], [0], color=COLORS[s], lw=2, label=s.title())
    for s in SPLITS
] + [
    Line2D([0], [0], color=COLORS["history"], lw=2.5, label="Observed history")
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
ordered_metrics = [
    "oil_cum", "gas_cum", "water_cum",
    "oil_rate", "gas_rate", "water_rate",
]

for ax, metric in zip(axes.flat, ordered_metrics):
    divisor = 1e6 if metric.endswith("_cum") else 1
    for case_num in unc["case_num"]:
        series = curves.get((int(case_num), metric))
        if series is None or series.empty:
            continue
        thin = sample_curve(series)
        ax.plot(
            thin.index, thin / divisor,
            color=COLORS[split_for_case(case_num)],
            alpha=0.23, linewidth=0.8,
        )

    observed = curves.get((0, metric))
    if observed is not None and not observed.empty:
        thin = sample_curve(observed)
        ax.plot(
            thin.index, thin / divisor,
            color=COLORS["history"], linewidth=2.4, zorder=10,
        )

    if metric.endswith("_rate"):
        quarterly = curves.get((0, metric + "_quarterly"))
        if quarterly is not None and not quarterly.empty:
            ax.scatter(
                quarterly.index, quarterly,
                s=22, facecolor="white", edgecolor="black", zorder=11,
            )

    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylabel(LABELS[metric])
    ax.set_xlabel("Date")

fig.legend(
    handles=legend_handles, loc="upper center",
    ncol=4, bbox_to_anchor=(0.5, 0.96),
)
fig.suptitle("Production response across 100 simulation cases", fontsize=18, y=1.01)
fig.tight_layout(rect=[0, 0, 1, 0.91])
show_save(fig, "05_production_curve_families")

In [0]:
CUM_METRICS = ["oil_cum", "gas_cum", "water_cum"]

required_keys = [
    (int(case_num), metric)
    for case_num in unc["case_num"]
    for metric in CUM_METRICS
]
missing_keys = [
    key for key in required_keys
    if key not in curves or curves[key].empty
]
if missing_keys:
    raise ValueError(f"Missing cumulative series; examples: {missing_keys[:10]}")

common_start = max(curves[key].index.min() for key in required_keys)
common_end = min(curves[key].index.max() for key in required_keys)
print(f"Shared cumulative coverage: {common_start} -> {common_end}")

endpoint_rows = []
for case_num in unc["case_num"]:
    row = {"case_num": int(case_num), "endpoint_date": common_end}
    for metric in CUM_METRICS:
        row[metric] = interpolate_inside(
            curves[(int(case_num), metric)], [common_end]
        )[0]
    endpoint_rows.append(row)

endpoints = unc.merge(
    pd.DataFrame(endpoint_rows), on="case_num", validate="one_to_one"
)
display(endpoints[CUM_METRICS].describe().T)
save_table(endpoints, "case_cumulative_endpoints")

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, subset_name, subset in [
    (axes[0], "All 100 cases", endpoints),
    (axes[1], "Training cases only", endpoints.query("split == 'train'")),
]:
    corr = subset[PARAMS + CUM_METRICS].corr(
        method="spearman"
    ).loc[PARAMS, CUM_METRICS]
    sns.heatmap(
        corr, annot=True, fmt=".2f", cmap="vlag",
        vmin=-1, vmax=1, ax=ax,
        cbar_kws={"label": "Spearman correlation"},
    )
    ax.set_title(subset_name)
    save_table(corr, f"parameter_endpoint_spearman_{subset_name.split()[0].lower()}",
               index=True)
fig.tight_layout()
show_save(fig, "06_parameter_endpoint_correlations")

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for row_idx, metric in enumerate(CUM_METRICS):
    for col_idx, param in enumerate(PARAMS):
        ax = axes[row_idx, col_idx]
        for split in SPLITS:
            subset = endpoints.loc[endpoints["split"] == split]
            ax.scatter(
                subset[param], subset[metric] / 1e6,
                color=COLORS[split], s=30, alpha=0.8,
            )
        ax.set_xlabel(param)
        ax.set_ylabel(LABELS[metric])
fig.suptitle(
    f"Uncertainty inputs versus cumulative production at {common_end:%Y-%m-%d}",
    fontsize=17,
)
fig.tight_layout()
show_save(fig, "07_parameter_endpoint_scatter")

In [0]:
rate_metrics = ["oil_rate", "gas_rate", "water_rate"]
history_rows = []

for case_num in unc["case_num"]:
    row = {"case_num": int(case_num)}
    for metric in rate_metrics:
        observed = curves.get((0, metric + "_quarterly"))
        if observed is None or observed.empty:
            observed = curves.get((0, metric))
            if observed is not None:
                observed = sample_curve(observed, max_points=1000)

        simulated = curves.get((int(case_num), metric))
        if observed is None or simulated is None:
            row[f"{metric}_nrmse"] = np.nan
            row[f"{metric}_n_points"] = 0
            continue

        prediction = interpolate_inside(simulated, observed.index)
        truth = observed.to_numpy(dtype=float)
        valid = np.isfinite(prediction) & np.isfinite(truth)
        row[f"{metric}_n_points"] = int(valid.sum())

        if valid.sum() < 3:
            row[f"{metric}_nrmse"] = np.nan
            continue

        normalizer = np.sqrt(np.mean(truth[valid] ** 2))
        rmse = np.sqrt(np.mean((prediction[valid] - truth[valid]) ** 2))
        row[f"{metric}_nrmse"] = (
            rmse / normalizer if normalizer > 0 else np.nan
        )
    history_rows.append(row)

history_scores = unc.merge(
    pd.DataFrame(history_rows), on="case_num", validate="one_to_one"
)
score_columns = [f"{metric}_nrmse" for metric in rate_metrics]
history_scores["history_score"] = history_scores[score_columns].mean(
    axis=1, skipna=False
)
history_scores = history_scores.sort_values("history_score")
save_table(history_scores, "diagnostic_history_match_scores")

display(history_scores[
    ["case_num", "split", "history_score"] + score_columns + PARAMS
].head(15))

valid_scores = history_scores.dropna(subset=["history_score"])
if len(valid_scores):
    top = valid_scores.head(10).copy()
    top["case_label"] = "Case " + top["case_num"].astype(str)

    fig, ax = plt.subplots(figsize=(11, 5))
    sns.barplot(
        data=top, x="case_label", y="history_score",
        hue="split", palette=COLORS, dodge=False, ax=ax,
    )
    ax.set(
        title="Ten closest history matches",
        xlabel="Simulation case",
        ylabel="Mean normalized RMSE (lower is better)",
    )
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    show_save(fig, "08_diagnostic_history_match_ranking")

    best_case = int(top.iloc[0]["case_num"])
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, metric in zip(axes, rate_metrics):
        simulation = sample_curve(curves[(best_case, metric)])
        ax.plot(
            simulation.index, simulation, color=COLORS["train"],
            lw=2, label=f"Case {best_case}",
        )
        observed = curves.get((0, metric + "_quarterly"))
        if observed is None or observed.empty:
            observed = curves.get((0, metric))
        if observed is not None:
            observed = sample_curve(observed)
            ax.plot(
                observed.index, observed, color="black",
                marker="o", markersize=3, lw=1.5, label="Observed",
            )
        ax.set_ylabel(LABELS[metric])
        ax.set_title(metric.replace("_", " ").title())
        ax.legend()
    fig.tight_layout()
    show_save(fig, "09_best_diagnostic_history_match")

print("This ranking is diagnostic only — not a future-forecast accuracy estimate.")

In [0]:
anomalies = production_audit.loc[
    (production_audit["invalid_value_rows"] > 0)
    | (production_audit["conflicting_timestamps"] > 0)
    | (production_audit["negative_values"] > 0)
    | (production_audit["cumulative_decreases"] > 0)
]
display(anomalies)
save_table(anomalies, "production_anomalies")

derived_rows = []
for case_num in unc["case_num"]:
    oil = curves.get((int(case_num), "oil_rate"))
    gas = curves.get((int(case_num), "gas_rate"))
    water = curves.get((int(case_num), "water_rate"))
    if oil is None or gas is None or water is None:
        continue

    dates = sample_curve(oil, max_points=250).index
    q_oil = interpolate_inside(oil, dates)
    q_gas = interpolate_inside(gas, dates)
    q_water = interpolate_inside(water, dates)
    producing = q_oil > 1e-8
    total_liquid = q_oil + q_water

    gor = np.full(len(dates), np.nan)
    water_cut = np.full(len(dates), np.nan)
    np.divide(q_gas, q_oil, out=gor, where=producing)
    np.divide(
        q_water, total_liquid, out=water_cut,
        where=total_liquid > 1e-8,
    )

    derived_rows.append(pd.DataFrame({
        "date": dates,
        "case_num": int(case_num),
        "split": split_for_case(case_num),
        "GOR": gor,
        "water_cut": water_cut,
    }))

derived_production = pd.concat(derived_rows, ignore_index=True)
save_table(derived_production, "sampled_gor_water_cut")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for case_num, group in derived_production.groupby("case_num"):
    color = COLORS[split_for_case(case_num)]
    axes[0].plot(group["date"], group["GOR"], color=color, alpha=0.25, lw=0.8)
    axes[1].plot(group["date"], group["water_cut"], color=color, alpha=0.25, lw=0.8)
axes[0].set(
    title="Produced gas-oil ratio",
    xlabel="Date", ylabel="Gas / oil (MSCF/STB)",
)
axes[1].set(
    title="Produced water cut",
    xlabel="Date", ylabel="Water / total liquid",
    ylim=(0, 1),
)
fig.tight_layout()
show_save(fig, "10_gor_and_water_cut")

In [0]:
grid_schema = pq.read_schema(GRID_PATH).names

def resolve_grid_column(token):
    if token in grid_schema:
        return token
    exact_abbreviation = [
        name for name in grid_schema
        if re.search(rf"\({re.escape(token)}\)\s*$", name, flags=re.I)
    ]
    if len(exact_abbreviation) == 1:
        return exact_abbreviation[0]
    normalized = {
        re.sub(r"[^a-z0-9]", "", name.lower()): name for name in grid_schema
    }
    key = re.sub(r"[^a-z0-9]", "", token.lower())
    if key in normalized:
        return normalized[key]
    raise KeyError(f"Cannot resolve {token!r}; available columns: {grid_schema}")

TOKENS = [
    "I Index", "J Index", "K Index",
    "X Coordinate", "Y Coordinate", "Z Coordinate",
    "PORV", "PORO", "PERMX", "PERMZ",
    "TRANX", "TRANY", "TRANZ",
    "FIPNUM", "PVTNUM", "SWL", "SWCR", "SGL", "SGCR", "SOWCR",
    "PRESSURE", "SWAT", "SOIL",
]
GRID_MAP = {}
for token in TOKENS:
    try:
        GRID_MAP[token] = resolve_grid_column(token)
    except KeyError:
        if token == "SGL":
            continue
        raise

def load_grid_case(case_num):
    columns = list(dict.fromkeys(GRID_MAP.values()))
    frame = pd.read_parquet(
        GRID_PATH,
        columns=columns,
        filters=[("case_num", "==", int(case_num))],
    )
    return frame.rename(columns={v: k for k, v in GRID_MAP.items()})

REFERENCE_CASE = int(unc["case_num"].min())
reference_grid = load_grid_case(REFERENCE_CASE)
INDEX_COLS = ["I Index", "J Index", "K Index"]
GRID_PROPS = [
    token for token in GRID_MAP
    if token not in INDEX_COLS
    and token not in ["X Coordinate", "Y Coordinate", "Z Coordinate"]
]
PROFILE_PROPS = [
    "PORO", "PERMX", "PERMZ", "PRESSURE", "SWAT", "SOIL",
    "PORV", "TRANX", "TRANY", "TRANZ",
]
reference_indexed = reference_grid.set_index(INDEX_COLS).sort_index()

feature_rows = []
quality_rows = []
layer_rows = []
sample_frames = []
comparison_rows = []

for case_num in unc["case_num"]:
    case_num = int(case_num)
    grid = reference_grid if case_num == REFERENCE_CASE else load_grid_case(case_num)
    grid[GRID_PROPS] = grid[GRID_PROPS].apply(pd.to_numeric, errors="coerce")
    numeric = grid[GRID_PROPS].to_numpy(dtype=float)

    sentinel_count = int((numeric == -999.25).sum())
    nonfinite_count = int((~np.isfinite(numeric)).sum())
    grid[GRID_PROPS] = grid[GRID_PROPS].replace(-999.25, np.nan)
    grid[GRID_PROPS] = grid[GRID_PROPS].replace([np.inf, -np.inf], np.nan)

    duplicate_cells = int(grid.duplicated(INDEX_COLS).sum())
    quality_rows.append({
        "case_num": case_num,
        "active_rows": len(grid),
        "duplicate_cell_indices": duplicate_cells,
        "remaining_sentinel_values": sentinel_count,
        "nonfinite_property_values": nonfinite_count,
        "porosity_outside_0_1": int(
            ((grid["PORO"] < 0) | (grid["PORO"] > 1)).sum()
        ),
        "negative_permeability": int((grid["PERMX"] < 0).sum()),
        "negative_pore_volume": int((grid["PORV"] < 0).sum()),
        "invalid_saturation_sum": int(
            ((grid["SWAT"] + grid["SOIL"] - 1).abs() > 1e-5).sum()
        ),
    })

    feature = {"case_num": case_num, "active_cells": len(grid)}
    for prop in GRID_PROPS:
        values = grid[prop].dropna()
        if values.empty:
            continue
        feature.update({
            f"{prop}_mean": values.mean(),
            f"{prop}_std": values.std(ddof=1),
            f"{prop}_min": values.min(),
            f"{prop}_max": values.max(),
            f"{prop}_nunique": values.nunique(),
        })
        if prop in PROFILE_PROPS:
            for quantile, label in [(0.1, "p10"), (0.5, "p50"), (0.9, "p90")]:
                feature[f"{prop}_{label}"] = values.quantile(quantile)

    pv = grid["PORV"]
    feature["PORV_total"] = pv.sum(min_count=1)
    feature["oil_reservoir_volume_proxy"] = (pv * grid["SOIL"]).sum(min_count=1)
    feature["water_reservoir_volume_proxy"] = (pv * grid["SWAT"]).sum(min_count=1)

    for prop in ["PORO", "PERMX", "PERMZ", "PRESSURE", "SWAT", "SOIL"]:
        mask = pv.notna() & grid[prop].notna() & (pv >= 0)
        denominator = pv.loc[mask].sum()
        feature[f"{prop}_pv_weighted"] = (
            (pv.loc[mask] * grid.loc[mask, prop]).sum() / denominator
            if denominator > 0 else np.nan
        )

    positive_perm = grid.loc[grid["PERMX"] > 0, "PERMX"]
    feature["PERMX_geometric_mean"] = (
        np.exp(np.log(positive_perm).mean()) if len(positive_perm) else np.nan
    )

    layer = grid.groupby("K Index").agg(
        active_cells=("PORO", "size"),
        PORV_total=("PORV", "sum"),
        PORO_mean=("PORO", "mean"),
        PERMX_mean=("PERMX", "mean"),
        PRESSURE_mean=("PRESSURE", "mean"),
        SWAT_mean=("SWAT", "mean"),
    ).reset_index()
    layer["case_num"] = case_num
    layer_rows.append(layer)

    feature_rows.append(feature)

    sample = grid[PROFILE_PROPS].sample(
        n=min(1500, len(grid)), random_state=SEED
    ).copy()
    sample["case_num"] = case_num
    sample["split"] = split_for_case(case_num)
    sample_frames.append(sample)

    if duplicate_cells == 0 and not reference_indexed.index.has_duplicates:
        indexed = grid.set_index(INDEX_COLS).sort_index()
        same_cells = indexed.index.equals(reference_indexed.index)
        comparison = {"case_num": case_num, "same_active_cells": same_cells}
        if same_cells:
            for prop in ["PORO", "PERMX", "PORV", "PRESSURE", "SWAT", "SOIL"]:
                reference_values = reference_indexed[prop].to_numpy(dtype=float)
                case_values = indexed[prop].to_numpy(dtype=float)
                mask = (
                    np.isfinite(reference_values)
                    & np.isfinite(case_values)
                    & (np.abs(reference_values) > 1e-12)
                    & (reference_values != -999.25)
                )
                ratio = case_values[mask] / reference_values[mask]
                comparison[f"{prop}_cell_ratio_median"] = (
                    np.median(ratio) if len(ratio) else np.nan
                )
                comparison[f"{prop}_cell_ratio_std"] = (
                    np.std(ratio) if len(ratio) else np.nan
                )
        comparison_rows.append(comparison)

    if case_num % 10 == 0:
        print(f"Profiled through Case {case_num}")

grid_features = pd.DataFrame(feature_rows)
grid_quality = pd.DataFrame(quality_rows)
layer_features = pd.concat(layer_rows, ignore_index=True)
grid_sample = pd.concat(sample_frames, ignore_index=True)
grid_comparison = pd.DataFrame(comparison_rows)

for name, frame in {
    "grid_case_features": grid_features,
    "grid_quality": grid_quality,
    "grid_layer_features": layer_features,
    "grid_matched_cell_comparison": grid_comparison,
}.items():
    save_table(frame, name)

grid_features.to_parquet(EDA_DIR / "grid_case_features.parquet", index=False)
layer_features.to_parquet(EDA_DIR / "grid_layer_features.parquet", index=False)

display(grid_quality.describe().T)
display(grid_features.head())
print("Grid profiling complete.")

In [0]:
PLOT_PROPERTIES = ["PORO", "PERMX", "PRESSURE", "SWAT", "SOIL", "PORV"]
PROPERTY_LABELS = {
    "PORO": "Porosity (fraction)",
    "PERMX": "Horizontal permeability (mD)",
    "PRESSURE": "Initial pressure (psi)",
    "SWAT": "Initial water saturation",
    "SOIL": "Initial oil saturation",
    "PORV": "Pore volume (reservoir barrels)",
}

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, prop in zip(axes.flat, PLOT_PROPERTIES):
    plot_data = grid_sample[[prop, "split"]].dropna()
    if prop in ["PERMX", "PORV"]:
        plot_data = plot_data.loc[plot_data[prop] > 0].copy()
        plot_data[prop] = np.log10(plot_data[prop])
        xlabel = f"log10 {PROPERTY_LABELS[prop]}"
    else:
        xlabel = PROPERTY_LABELS[prop]

    sns.histplot(
        data=plot_data, x=prop, hue="split",
        hue_order=SPLITS, palette=COLORS,
        stat="density", common_norm=False,
        bins=45, element="step", fill=False, ax=ax,
    )
    ax.set_title(prop)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Sample density")
fig.suptitle("Grid property heterogeneity", fontsize=17)
fig.tight_layout()
show_save(fig, "11_grid_property_distributions")

case_grid = unc.merge(grid_features, on="case_num", validate="one_to_one")
summary_columns = [
    "PORO_mean", "PERMX_geometric_mean", "PRESSURE_pv_weighted",
    "SWAT_pv_weighted", "PORV_total", "oil_reservoir_volume_proxy",
]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, feature in zip(axes.flat, summary_columns):
    sns.boxplot(
        data=case_grid, x="split", y=feature,
        order=SPLITS, color="#E8E8E8", ax=ax,
    )
    sns.stripplot(
        data=case_grid, x="split", y=feature,
        order=SPLITS, hue="split", palette=COLORS,
        size=4, alpha=0.7, ax=ax,
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_title(feature.replace("_", " "))
    ax.set_xlabel("")
    ax.set_ylabel("Case-level value")
fig.tight_layout()
show_save(fig, "12_grid_case_summary_split_comparison")

In [0]:
spatial = reference_grid.copy()
spatial[GRID_PROPS] = spatial[GRID_PROPS].replace(-999.25, np.nan)

layer_values = np.sort(spatial["K Index"].dropna().unique())
selected_k = layer_values[len(layer_values) // 2]
layer = spatial.loc[spatial["K Index"] == selected_k]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, prop, cmap in zip(
    axes, ["PORO", "PERMX", "SWAT"], ["viridis", "magma", "Blues"]
):
    values = layer[prop].copy()
    label = PROPERTY_LABELS[prop]
    if prop == "PERMX":
        values = np.log10(values.where(values > 0))
        label = "log10 horizontal permeability (mD)"

    map_frame = layer[["I Index", "J Index"]].copy()
    map_frame["value"] = values
    matrix = map_frame.pivot_table(
        index="J Index", columns="I Index", values="value", aggfunc="mean"
    )
    image = ax.pcolormesh(
        matrix.columns.to_numpy(),
        matrix.index.to_numpy(),
        matrix.to_numpy(),
        shading="nearest", cmap=cmap,
    )
    ax.set(
        title=f"{prop} at K = {selected_k:g}",
        xlabel="I index", ylabel="J index",
    )
    ax.set_aspect("equal")
    fig.colorbar(image, ax=ax, label=label, shrink=0.8)
fig.suptitle(f"Case {REFERENCE_CASE}: horizontal reservoir slice", fontsize=17)
fig.tight_layout()
show_save(fig, "13_horizontal_grid_maps")

j_values = np.sort(spatial["J Index"].dropna().unique())
selected_j = j_values[len(j_values) // 2]
section = spatial.loc[spatial["J Index"] == selected_j]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, prop, cmap in zip(
    axes, ["PORO", "PERMX", "SWAT"], ["viridis", "magma", "Blues"]
):
    section_frame = section[["I Index", "K Index", prop]].copy()
    if prop == "PERMX":
        section_frame[prop] = np.log10(
            section_frame[prop].where(section_frame[prop] > 0)
        )
    matrix = section_frame.pivot_table(
        index="K Index", columns="I Index", values=prop, aggfunc="mean"
    )
    image = ax.pcolormesh(
        matrix.columns.to_numpy(),
        matrix.index.to_numpy(),
        matrix.to_numpy(),
        shading="nearest", cmap=cmap,
    )
    ax.invert_yaxis()
    ax.set(
        title=f"{prop} at J = {selected_j:g}",
        xlabel="I index", ylabel="K layer index",
    )
    fig.colorbar(image, ax=ax, shrink=0.8)
fig.suptitle(
    f"Case {REFERENCE_CASE}: vertical cross-section",
    fontsize=17,
)
fig.tight_layout()
show_save(fig, "14_vertical_cross_sections")

footprint = spatial.groupby(["I Index", "J Index"]).agg(
    x=("X Coordinate", "mean"),
    y=("Y Coordinate", "mean"),
    active_layers=("K Index", "nunique"),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 7))
points = ax.scatter(
    footprint["x"], footprint["y"],
    c=footprint["active_layers"], s=12, cmap="viridis",
)
ax.set(
    title=f"Case {REFERENCE_CASE}: active reservoir footprint",
    xlabel="X coordinate (m)", ylabel="Y coordinate (m)",
)
ax.set_aspect("equal")
ax.ticklabel_format(style="plain", useOffset=False)
fig.colorbar(points, ax=ax, label="Number of active layers")
fig.tight_layout()
show_save(fig, "15_reservoir_footprint")

reference_layers = layer_features.query("case_num == @REFERENCE_CASE")
fig, axes = plt.subplots(1, 3, figsize=(14, 6))
for ax, feature in zip(axes, ["PORV_total", "PERMX_mean", "SWAT_mean"]):
    ax.plot(
        reference_layers[feature], reference_layers["K Index"],
        marker="o", lw=1.5, color=COLORS["train"],
    )
    ax.invert_yaxis()
    ax.set_xlabel(feature.replace("_", " "))
    ax.set_ylabel("K layer index")
    ax.set_title(feature.replace("_", " "))
fig.tight_layout()
show_save(fig, "16_layer_profiles")

In [0]:
redundancy_features = [
    "PORO_mean", "PERMX_geometric_mean", "PORV_total",
    "PRESSURE_pv_weighted", "SWAT_pv_weighted",
    "oil_reservoir_volume_proxy",
]
train_case_grid = case_grid.query("split == 'train'")

correlation = train_case_grid[
    PARAMS + redundancy_features
].corr(method="spearman").loc[PARAMS, redundancy_features]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    correlation, cmap="vlag", vmin=-1, vmax=1,
    annot=True, fmt=".2f", ax=ax,
    cbar_kws={"label": "Training-case Spearman correlation"},
)
ax.set_title("Are grid summaries alternative encodings of uncertainty inputs?")
fig.tight_layout()
show_save(fig, "17_grid_input_redundancy")
save_table(correlation, "training_grid_input_correlations", index=True)

comparison_columns = [
    column for column in grid_comparison
    if column.endswith("_cell_ratio_std")
]
display(grid_comparison[comparison_columns].describe().T)

In [0]:
from scipy.linalg import svd

time_grid = pd.date_range(common_start, common_end, periods=160)
train_case_numbers = unc.loc[
    unc["split"] == "train", "case_num"
].astype(int).tolist()

pca_rows = []
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, CUM_METRICS):
    matrix = np.vstack([
        interpolate_inside(curves[(case_num, metric)], time_grid)
        for case_num in train_case_numbers
    ])
    if not np.isfinite(matrix).all():
        raise ValueError(f"Nonfinite aligned training curves for {metric}")

    centered = matrix - matrix.mean(axis=0, keepdims=True)
    _, singular_values, _ = svd(centered, full_matrices=False)
    variance = singular_values ** 2
    if variance.sum() > 0:
        explained = variance / variance.sum()
        cumulative_explained = np.cumsum(explained)
        k_99 = int(np.searchsorted(cumulative_explained, 0.99) + 1)
    else:
        explained = np.zeros_like(variance)
        cumulative_explained = explained
        k_99 = 0

    components = np.arange(1, len(explained) + 1)
    ax.plot(
        components, cumulative_explained,
        marker="o", markersize=3, color=COLORS["train"],
    )
    ax.axhline(0.99, color="black", linestyle="--", lw=1)
    ax.set(
        title=f"{metric}: {k_99} components for 99%",
        xlabel="Number of components",
        ylabel="Cumulative explained variance",
        ylim=(0, 1.02),
        xlim=(1, min(20, len(components))),
    )
    pca_rows.append({
        "metric": metric,
        "training_cases": len(train_case_numbers),
        "aligned_timestamps": len(time_grid),
        "components_for_99_percent": k_99,
        "first_component_fraction": explained[0] if len(explained) else np.nan,
    })

fig.suptitle("Dimensionality of training-case cumulative curve variation", fontsize=17)
fig.tight_layout()
show_save(fig, "19_training_curve_dimensionality")
pca_report = pd.DataFrame(pca_rows)
display(pca_report)
save_table(pca_report, "training_curve_dimensionality")

print(
    "Low-rank curves support PCA/basis-coefficient surrogates. "
    "Future forecasting still needs a constrained temporal model."
)

In [0]:
modeling_candidates = (
    unc.merge(grid_features, on="case_num", validate="one_to_one")
       .merge(
           pd.DataFrame(endpoint_rows),
           on="case_num", validate="one_to_one",
       )
)
modeling_candidates.to_parquet(
    EDA_DIR / "case_modeling_candidates.parquet", index=False
)

quality_summary = {
    "uncertainty_cases": int(len(unc)),
    "split_counts": {
        str(key): int(value)
        for key, value in unc["split"].value_counts().items()
    },
    "production_raw_rows": int(raw_prod_rows),
    "grid_active_counts": [
        int(value) for value in sorted(grid_quality["active_rows"].unique())
    ],
    "shared_history_start": str(common_start),
    "shared_history_end": str(common_end),
    "figure_directory": str(FIG_DIR),
    "table_directory": str(TABLE_DIR),
}

with open(EDA_DIR / "quality_summary.json", "w", encoding="utf-8") as handle:
    json.dump(quality_summary, handle, indent=2)

(EDA_DIR / "MODELING_HANDOFF.md").write_text(f"""# EDA Modeling Handoff

## Dataset Dimensions
- Uncertainty cases: {len(unc)}
- Raw production rows: {raw_prod_rows:,}
- Grid active rows per case: {quality_summary['grid_active_counts']}
- Shared history coverage: {common_start} to {common_end}

## Recommended Baseline Sequence
1. Fit input scaling using Cases 1-70 only.
2. Start with the four uncertainty parameters as inputs.
3. Compare regularized regression and GP surrogates for curve coefficients.
4. Add grid summaries only if they improve validation results.
5. Validate temporal forecasting with rolling history cutoffs.
6. Use Cases 71-85 for model selection; evaluate Cases 86-100 once.

## Exported Artifacts
- case_modeling_candidates.parquet
- grid_case_features.parquet
- grid_layer_features.parquet
- quality_summary.json
- tables/*.csv
- figures/*.png and figures/*.pdf
""", encoding="utf-8")

display(pd.DataFrame(
    [(key, str(value)) for key, value in quality_summary.items()],
    columns=["check", "result"],
))
print(f"EDA complete. Outputs saved to {EDA_DIR}")